# Fire-progression exclusion criterion: measure what was actually excluded

Runs on the TS-SatFire dataset only. No checkpoints, no GPU.
Expected runtime: 10-15 minutes.

The previous audit measured cumulative burned area. The fire-progression label
is the day-to-day increment, BA(T+1) minus BA(T), so the exclusion criterion in
Section 4.3 has to be checked against that quantity, not against cumulative burn.

This notebook answers:
  1. What is the true progression-pixel fraction for each of the 24 test fires?
  2. Does any single stated criterion select exactly the eight fires we excluded?
  3. Which AF test fire is only partially labelled?
  4. Produces the per-fire F1 vs crop-truncation figure for Section 6.3.


In [ ]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor

warnings.filterwarnings("ignore")

try:
    import rasterio
    HAS_RASTERIO = True
except Exception:
    import tifffile
    HAS_RASTERIO = False

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)
CROP = 256
TS = 2  # manuscript fire-progression temporal window


def find_data_root():
    for p in ["/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire",
              "/kaggle/input/ts-satfire/ts-satfire/ts-satfire",
              "/kaggle/input/ts-satfire/ts-satfire",
              "/kaggle/input/ts-satfire"]:
        if os.path.isdir(p):
            return p
    return None


DATA_ROOT = find_data_root()
assert DATA_ROOT, "TS-SatFire dataset not found"
print("DATA_ROOT:", DATA_ROOT)

BAFP_TEST = sorted(d for d in os.listdir(DATA_ROOT)
                   if d.startswith("US_2021")
                   and os.path.isdir(os.path.join(DATA_ROOT, d)))
print("BA/FP test fires:", len(BAFP_TEST))

# The eight fires excluded from the fire-progression test aggregate.
FP_TEST_EXCLUDE = [
    "US_2021_FL2521008104520210308",
    "US_2021_MT4714310953420211004",
    "US_2021_NM3323810847220210520",
    "US_2021_NM3340210587120210426",
    "US_2021_NM3344410803520210514",
    "US_2021_NM3676810505920211120",
    "US_2021_AZ3345510938920210616",
    "US_2021_AZ3368910927620210616",
]

AF_TEST = [
    "elephant_hill_fire", "eagle_bluff_fire", "double_creek_fire", "sparks_lake_fire",
    "lytton_fire", "chuckegg_creek_fire", "swedish_fire", "sydney_fire",
    "thomas_fire", "tubbs_fire", "carr_fire", "camp_fire",
    "creek_fire", "blue_ridge_fire", "dixie_fire", "mosquito_fire", "calfcanyon_fire",
]


## 1. Measure progression pixels per fire

A window is usable when both the last input day and the following day carry a
band-8 raster with at least one finite pixel, matching Section 3.1.
The progression label is `isfinite(band8[T+1]) AND NOT isfinite(band8[T])`.


In [ ]:
def read_band(path, idx1):
    if HAS_RASTERIO:
        with rasterio.open(path) as src:
            if src.count < idx1:
                return None
            return src.read(idx1).astype(np.float32)
    arr = tifffile.imread(path).astype(np.float32)
    if arr.ndim == 2:
        arr = arr[np.newaxis]
    return arr[idx1 - 1] if arr.shape[0] >= idx1 else None


def crop_center(a, size=CROP):
    h, w = a.shape
    r0, c0 = max((h - size) // 2, 0), max((w - size) // 2, 0)
    return a[r0:r0 + size, c0:c0 + size]


def measure_progression(fid):
    fdir = os.path.join(DATA_ROOT, fid)
    files = sorted(glob.glob(os.path.join(fdir, "VIIRS_Day", "*.tif")))
    rec = {"fire": fid, "n_days": len(files), "n_windows_possible": 0,
           "n_windows_usable": 0, "n_windows_nonzero": 0,
           "prog_px_in": 0, "prog_px_full": 0,
           "max_window_frac_pct": 0.0, "error": None}
    if len(files) < TS + 1:
        return rec

    masks = []
    try:
        for f in files:
            b8 = read_band(f, 8)
            masks.append(None if b8 is None else np.isfinite(b8))
    except Exception as e:
        rec["error"] = "{}: {}".format(type(e).__name__, e)
        return rec

    for t0 in range(len(files) - TS):
        rec["n_windows_possible"] += 1
        last, nxt = masks[t0 + TS - 1], masks[t0 + TS]
        if last is None or nxt is None:
            continue
        if not last.any() or not nxt.any():
            continue  # a day with no BA label at all
        rec["n_windows_usable"] += 1

        prog_full = nxt & ~last
        prog_in = crop_center(nxt) & ~crop_center(last)
        n_in = int(prog_in.sum())
        rec["prog_px_in"] += n_in
        rec["prog_px_full"] += int(prog_full.sum())
        if n_in > 0:
            rec["n_windows_nonzero"] += 1
        frac = 100.0 * n_in / (CROP * CROP)
        rec["max_window_frac_pct"] = max(rec["max_window_frac_pct"], frac)
    return rec


print("Measuring progression for", len(BAFP_TEST), "fires...")
with ThreadPoolExecutor(max_workers=8) as ex:
    recs = list(ex.map(measure_progression, BAFP_TEST))

fp = pd.DataFrame(recs)
fp["mean_window_frac_pct"] = np.where(
    fp.n_windows_usable > 0,
    100.0 * fp.prog_px_in / (fp.n_windows_usable * CROP * CROP), np.nan)
fp["prog_outside_pct"] = np.where(
    fp.prog_px_full > 0, 100.0 * (1 - fp.prog_px_in / fp.prog_px_full), np.nan)
fp["excluded_in_manuscript"] = fp.fire.isin(FP_TEST_EXCLUDE)
fp = fp.sort_values("mean_window_frac_pct", na_position="first")
fp.to_csv(os.path.join(OUT, "fp_progression_measure.csv"), index=False)

cols = ["fire", "n_days", "n_windows_usable", "n_windows_nonzero",
        "mean_window_frac_pct", "max_window_frac_pct", "prog_outside_pct",
        "excluded_in_manuscript"]
print(fp[cols].to_string(index=False,
                         float_format=lambda x: "{:.4f}".format(x)))
print("\nErrors:", int(fp.error.notna().sum()))


## 2. Does any single criterion select exactly the eight excluded fires?


In [ ]:
print("Excluded set in the manuscript ({} fires):".format(len(FP_TEST_EXCLUDE)))
for f in sorted(FP_TEST_EXCLUDE):
    r = fp[fp.fire == f]
    if len(r):
        r = r.iloc[0]
        print("  {:<32} mean {:.4f}%  max {:.4f}%  usable_win {:>3d}  nonzero_win {:>3d}".format(
            f[8:], r.mean_window_frac_pct, r.max_window_frac_pct,
            int(r.n_windows_usable), int(r.n_windows_nonzero)))

print("\nCandidate criteria, and how many of the 24 they select:\n")
print("{:<44} {:>8} {:>10} {:>10}".format("criterion", "selects", "matches_8", "note"))
print("-" * 78)

target = set(FP_TEST_EXCLUDE)


def test_rule(name, mask):
    sel = set(fp.fire[mask].tolist())
    exact = (sel == target)
    note = ""
    if not exact:
        miss = target - sel
        extra = sel - target
        note = "miss {} extra {}".format(len(miss), len(extra))
    print("{:<44} {:>8d} {:>10} {:>10}".format(name, len(sel), str(exact), note))
    return sel, exact


rules = []
for thr in [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1]:
    rules.append(("mean window progression < {}%".format(thr),
                  fp.mean_window_frac_pct < thr))
for thr in [0.001, 0.01, 0.05, 0.1, 0.5]:
    rules.append(("max window progression < {}%".format(thr),
                  fp.max_window_frac_pct < thr))
rules.append(("zero usable windows", fp.n_windows_usable == 0))
rules.append(("zero windows with any progression", fp.n_windows_nonzero == 0))
for k in [1, 2, 3, 5]:
    rules.append(("fewer than {} nonzero windows".format(k), fp.n_windows_nonzero < k))

results = {}
for name, mask in rules:
    sel, exact = test_rule(name, mask)
    results[name] = {"selects": sorted(sel), "exact_match": bool(exact)}

json.dump(results, open(os.path.join(OUT, "fp_criterion_search.json"), "w"), indent=1)

exact_rules = [k for k, v in results.items() if v["exact_match"]]
print("\n" + "=" * 78)
if exact_rules:
    print("Criteria that reproduce the manuscript's eight exclusions exactly:")
    for k in exact_rules:
        print("   ", k)
    print("\nUse one of these verbatim in Section 4.3.")
else:
    print("NO single tested criterion reproduces the eight exclusions exactly.")
    print("The exclusion list must therefore be either:")
    print("  (a) restated with the criterion that does match, or")
    print("  (b) revised so that the fires excluded follow from a stated rule, or")
    print("  (c) justified fire by fire, with the numbers reported.")
    print("\nSmallest criterion covering all eight (may over-select):")
    for name, mask in rules:
        sel = set(fp.fire[mask].tolist())
        if target.issubset(sel):
            print("   {} -> selects {} fires ({} extra)".format(
                name, len(sel), len(sel - target)))
            break
print("=" * 78)


## 3. Which AF test fire is only partially labelled?


In [ ]:
def af_days(fid):
    files = sorted(glob.glob(os.path.join(DATA_ROOT, fid, "VIIRS_Day", "*.tif")))
    n_lab = 0
    n_pos = 0
    for f in files:
        b7 = read_band(f, 7)
        if b7 is None:
            continue
        if np.isfinite(b7).any():
            n_lab += 1
        n_pos += int((np.nan_to_num(b7, nan=0.0) >= 7).sum())
    return {"fire": fid, "n_days": len(files), "days_labelled": n_lab,
            "frac_days": (n_lab / len(files)) if files else 0.0,
            "af_pos_px": n_pos}


with ThreadPoolExecutor(max_workers=8) as ex:
    af = pd.DataFrame(list(ex.map(af_days, AF_TEST)))
af = af.sort_values("frac_days")
af.to_csv(os.path.join(OUT, "af_test_label_coverage.csv"), index=False)
print(af.to_string(index=False, float_format=lambda x: "{:.3f}".format(x)))

partial = af[(af.frac_days > 0) & (af.frac_days < 0.5)]
print("\nAF test fires labelled on fewer than half their days:")
if len(partial):
    for _, r in partial.iterrows():
        print("  {:<24} {}/{} days ({:.1f}%), {} positive pixels".format(
            r.fire, int(r.days_labelled), int(r.n_days),
            100 * r.frac_days, int(r.af_pos_px)))
    print("\nDisclose this in Section 4.1 and report its per-fire F1 separately.")
else:
    print("  none")

print("\nAF test fires with no labels at all:")
for _, r in af[af.days_labelled == 0].iterrows():
    print("  {}".format(r.fire))


## 4. Figure for Section 6.3: per-fire FP F1 against crop truncation


In [ ]:
# Published per-fire F1 (manuscript Figure 6), keyed to full fire IDs.
F1_BY_FIRE = {
    "US_2021_CA3604711863120210910": 0.532,
    "US_2021_ID4558511544420210705": 0.537,
    "US_2021_WA4856812048820210708": 0.518,
    "US_2021_CA3568711855020210818": 0.452,
    "US_2021_CA3658211879520210912": 0.451,
    "US_2021_ID4453211532920210810": 0.439,
    "US_2021_MT4568311385420210708": 0.433,
    "US_2021_WA4828511853120210713": 0.422,
    "US_2021_CA3451712013120211011": 0.373,
    "US_2021_MT4579011310120210708": 0.371,
    "US_2021_ID4663811466720210707": 0.365,
    "US_2021_WA4877811903420210803": 0.310,
    "US_2021_ID4762711608320210708": 0.208,
    "US_2021_CA3627811855020210815": 0.204,
    "US_2021_WA4879111827120210805": 0.153,
    "US_2021_CA4086312235520210630": 0.007,
}

sub = fp[fp.fire.isin(F1_BY_FIRE)].copy()
sub["f1"] = sub.fire.map(F1_BY_FIRE)
sub = sub.dropna(subset=["prog_outside_pct"])

if len(sub) >= 3:
    x = sub.prog_outside_pct.values
    y = sub.f1.values
    r = np.corrcoef(x, y)[0, 1]
    order_x = np.argsort(np.argsort(x)); order_y = np.argsort(np.argsort(y))
    rho = np.corrcoef(order_x, order_y)[0, 1]
    print("n = {}  Pearson r = {:.3f}  Spearman rho = {:.3f}".format(len(sub), r, rho))

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.scatter(x, y, s=45, c="tab:red", edgecolor="k", linewidth=0.5, zorder=3)
    for _, rrow in sub.iterrows():
        ax.annotate(rrow.fire[8:14], (rrow.prog_outside_pct, rrow.f1),
                    fontsize=6.5, xytext=(3, 3), textcoords="offset points")
    if len(sub) > 2:
        m, b = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 50)
        ax.plot(xs, m * xs + b, "k--", lw=1,
                label="linear fit (r = {:.2f})".format(r))
    ax.axhline(0.375, color="tab:blue", ls=":", lw=1.2,
               label="best published baseline (0.375)")
    ax.set_xlabel("Progression pixels outside the 256 x 256 crop (%)")
    ax.set_ylabel("Per-fire test F1")
    ax.set_title("Fire-progression accuracy against crop truncation")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, "fig_fp_f1_vs_crop.png"), dpi=200)
    plt.close()
    sub[["fire", "f1", "prog_outside_pct", "mean_window_frac_pct"]].to_csv(
        os.path.join(OUT, "fp_f1_vs_crop.csv"), index=False)
    print("Figure written: fig_fp_f1_vs_crop.png")
else:
    print("Not enough overlapping fires to plot.")


## 5. Files written


In [ ]:
for f in sorted(glob.glob(os.path.join(OUT, "*.csv")) +
                glob.glob(os.path.join(OUT, "*.json")) +
                glob.glob(os.path.join(OUT, "*.png"))):
    print("  {:>8.1f} KB  {}".format(os.path.getsize(f) / 1e3, f))
print("\nSend me the printed output of sections 2 and 3.")
